In [1]:
import pandas as pd
import numpy as np
import os

from pydantic_core.core_schema import filter_dict_schema

I've tried to design this notebook so that we can 'run all' once the following cell, which gives a list containing all the 'Transee' data.

In [2]:
#Adding functions to read and write the correct dtypes for the csv files 
# Source - https://stackoverflow.com/a/50051542
# Posted by Aaron Brock, modified by community. See post 'Timeline' for change history
# Retrieved 2026-02-26, License - CC BY-SA 3.0

def to_csv(df, path):
    # Prepend dtypes to the top of df (from https://stackoverflow.com/a/43408736/7607701)
    df.loc[-1] = df.dtypes
    df.index = df.index + 1
    df.sort_index(inplace=True)
    # Then save it to a csv
    df.to_csv(path, index=False)

def read_csv(path):
    # Read types first line of csv
    dtypes = {key:value for (key,value) in pd.read_csv(path,    
              nrows=1).iloc[0].to_dict().items() if 'date' not in value}

    parse_dates = [key for (key,value) in pd.read_csv(path, 
                   nrows=1).iloc[0].to_dict().items() if 'date' in value]
    # Read the rest of the lines with the types from above
    return pd.read_csv(path, dtype=dtypes, parse_dates=parse_dates, skiprows=[1])

In [3]:
# Dictionary to store the desired dtypes for each column
# The datetime objects will have to be dealt with seperately, so we add those to the list of columns below

# First I will list all of the columns that appear in the data/raw_data/schedule_data with their desired dtypes
# Commenting out the ones that are going to be dropped so that we can just avoid reading them in
schedule_dtypes ={
    #"Unnamed: 0": "int64",
    #"Time": "string",
    "Vehicle": "float64",
    "Gap": "object", # I would really like for this to be a string but this caused some issues later so I will try to read it in as an object instead or just not specify the dtype? 
    "Headway": "object",  
    "Schedule": "object", 
    "Destination": "object",
    #"day": "datetime64[us]",  # deal with date type seperately            
    "day of the week": "int64",
    #"Riders after stop": "float64"
}

# List all columns from dictionary keys plus any datetime columns
schedule_cols = ["day"] + list(schedule_dtypes.keys())

In [ ]:
# Next read the data from each csv with the desired columns
schedule_data_dir = '../data/raw_data/schedule_data/'
csv_files = [file for file in os.listdir(schedule_data_dir) if file[-4:] == '.csv']
dataframes = [pd.read_csv(schedule_data_dir + file, usecols=schedule_cols ) for file in csv_files ]


In [5]:
# Drop columns I'm not sure what to do with.

# Commenting out for now, this was dealt with above
#for df in dataframe:
#    for remove in ['Unnamed: 0', 'Time', 'Riders after stop']:
#        if remove in df.columns:
#            df.drop(remove, axis=1, inplace=True)

In [6]:
def find_datetime(day: str | float, time: str | float) -> pd.Timestamp | float:
    if type(day) != str or type(time) != str:
        return np.nan
    if time[0] == ' ':
        time = time[1:]
    return pd.Timestamp(day+' '+time)


In [7]:
for df in dataframes:
    df['Time'] = df.Schedule.str[-10:]
    df['scheduled time'] = df.apply(lambda x: find_datetime(day=x['day'], time=x['Time']), axis=1)
    df.drop(columns=['day', 'Time', 'day of the week'], inplace=True)

We're going to eventually split the data into eastbound vs westbound, so we'll add a column to address this.

In [8]:
def east_or_west(dest: str| float) -> str| float:
    if type(dest) != str:
        return np.nan
    dest = dest.lower()
    if 'east' in dest:
        return 'E'
    else:
        return 'W'


In [9]:
for df in dataframes:
    df['EB/WB'] = df['Destination'].apply(east_or_west)
    df.drop('Destination', axis=1, inplace=True)

In [10]:
def min_delay(schedule: str|float) -> int|float:
    if type(schedule) != str:
        return np.nan

    schedule = schedule.lower()
    min_marker = schedule.find(':')
    hour_marker = schedule[:min_marker].rfind(' ')
    if hour_marker == -1:
        hour_marker = 0

    minute = (int(schedule[hour_marker:min_marker])
              +int(schedule[min_marker+1:min_marker+3])/60)

    if 'ahead' in schedule:
        return -minute
    elif 'behind' in schedule:
        return minute
    else:
        return 0

for df in dataframes:
    df['min delay'] = df['Schedule'].apply(min_delay)
    df.drop('Schedule', axis=1, inplace=True)

Finally, 'cleaned_df' will be all of the dataframes merged together.

In [11]:
cleaned_df = pd.concat(dataframes, ignore_index=True)

In [12]:
cleaned_df.head()

,Vehicle,Gap,Headway,scheduled time,EB/WB,min delay
0,4574.0,NaN,8:20,2025-01-01 07:24:46,E,-2.600000
1,4552.0,12:51,10:00,2025-01-01 23:27:55,E,-1.883333
2,4591.0,7:46,10:00,2025-01-01 23:47:55,E,-14.133333
3,4575.0,16:24,10:00,2025-01-01 23:37:55,E,12.283333
4,4629.0,7:52,10:00,2025-01-01 23:57:55,E,0.150000


In [13]:
# We drop the instances where the min delay is over 2 hours, since that is most
# likely a cancellation rather than actual data.
cleaned_df = cleaned_df[cleaned_df['min delay'] < 120]

In [14]:
'12:00:00'.rfind(':')
'34:56'[-2:]

'56'

In [15]:
# Here we prune the data frame from segments where there is no gap info.
# First let's make sure that the Gap column is in time format.
# Turning the gap into a timestamp is problematic because it is sometimes negative.
# As well, sometimes the gap is sometimes a few hours (so it is of the format hh:mm:ss).
# Therefore, I elect to turn it into a float, where the units are in seconds.
def gap_to_seconds(gap: str|float) -> float:
    if type(gap) != str:
        return np.nan

    if gap.find('-') == -1:
        is_negative = False
    else:
        gap = gap[1:]
        is_negative = True

    if gap.count(':') > 1:
        hours = int(gap[:-6])
        minutes = int(gap[-5:-3])
        seconds = int(gap[-2:])
    else:
        hours = 0
        minutes = int(gap[:-3])
        seconds = int(gap[-2:])

    total_seconds = (hours*60 + minutes)*60 + seconds
    if is_negative:
        return -total_seconds
    else:
        return total_seconds



cleaned_df.Gap = cleaned_df.Gap.apply(gap_to_seconds)

n= 25
gap_na = cleaned_df.Gap.isna()
# For each index, this gives the largest size of a sequence of NaNs containing
# the index if the index is a NaN, and if the index is not a NaN, it gives the
# largest size of a sequence of non-NaNs.
s= gap_na.groupby(gap_na.diff().ne(0).cumsum()).transform('count')
# We only keep the data where the sequence of gaps of NaNs is smaller than n.
# If the Gap index is NaN, then the component is at most n.
cleaned_df = cleaned_df.loc[~(gap_na)|(s<=n)]

In [16]:
# I am going to elect to drop the times where there is no scheduling info
cleaned_df.dropna(axis=0, subset=['scheduled time'], inplace=True)

In [17]:
# Finally, we time order cleaned_df.
cleaned_df.sort_values(by=['scheduled time'], ignore_index=True, inplace=True)

In [18]:
cleaned_df.head()

,Vehicle,Gap,Headway,scheduled time,EB/WB,min delay
0,4418.0,NaN,10:00,2025-01-01 00:00:00,W,0.000000
1,4552.0,723.0,10:00,2025-01-01 00:00:17,E,-5.350000
2,4400.0,684.0,11:15,2025-01-01 00:00:39,W,-1.616667
3,4591.0,511.0,10:00,2025-01-01 00:00:40,E,-13.883333
4,4481.0,61.0,8:20,2025-01-01 00:00:49,W,11.933333


The goal is to use the TTC summary data as the basis of our dataset, with columns appended
from cleaned_df. First we need to add a 'EB/WB' column.

In [19]:
summary_df = pd.read_csv('../data/Munroe-Streetcar-Info/Streetcar_data_cleaned_up.csv')
remove_columns = ['Unnamed: 0', 'First dep NB or WB', 'First dep SB or EB', 'Last dep NB or WB', 'Last dep SB or EB', 'Route']
for column in remove_columns:
    if column in summary_df.columns:
        summary_df.drop(column, axis=1, inplace=True)

In [20]:
main_df_East = summary_df.copy()
main_df_East['EB/WB'] = ['E' for _ in range(main_df_East.shape[0])]
main_df_West = summary_df.copy()
main_df_West['EB/WB'] = ['W' for _ in range(main_df_West.shape[0])]

main_df = pd.concat([main_df_East, main_df_West], ignore_index=True)
main_df = main_df.sort_values(by=['time period start', 'time period end'], ignore_index=True)
main_df.head()

,date,No. of Veh,Service interval,Run time (min),Term time (min),Avg. spd (km/h),Time of Day (morning: 0-late evening: 4)*,Weekday=0/Sat=1/Sun=2,RT dist (km),Interruption,time period start,time period end,EB/WB
0,2023-01-14,12,10:00,106,14,18.4,0,1,32.56,NaN,2023-01-14 04:00:00,2023-01-14 08:00:00,E
1,2023-01-14,12,10:00,106,14,18.4,0,1,32.56,NaN,2023-01-14 04:00:00,2023-01-14 08:00:00,W
2,2023-01-14,16,09:30,138,14,14.2,1,1,32.56,NaN,2023-01-14 08:00:00,2023-01-14 12:00:00,E
3,2023-01-14,16,09:30,138,14,14.2,1,1,32.56,NaN,2023-01-14 08:00:00,2023-01-14 12:00:00,W
4,2023-01-14,20,08:30,156,14,12.5,2,1,32.56,NaN,2023-01-14 12:00:00,2023-01-14 19:00:00,E


In [21]:
# Let's turn the interruption 'nans' into 0-1s.
def binary_interruption(interruption: str|float) -> int:
    if type(interruption) != str:
        return 0
    else:
        return 1

main_df['Interruption'] = main_df['Interruption'].apply(binary_interruption)

# Turns the time periods into pandas timestamps
main_df['time period start'] = main_df['time period start'].apply(lambda x: pd.Timestamp(x))
main_df['time period end'] = main_df['time period end'].apply(lambda x: pd.Timestamp(x))

In [22]:
# We should filter main_df by the dates coming from cleaned_df.
earliest_timestamp = cleaned_df['scheduled time'].min()
latest_timestamp = cleaned_df['scheduled time'].max()
main_df = main_df[(main_df['time period start'] >= earliest_timestamp) & (main_df['time period end'] <= latest_timestamp)]

In [23]:
def count_bunch(df: pd.DataFrame) -> int:
    return len(df.Gap[df.Gap <= 120])

def count_gap(df: pd.DataFrame) -> int:
    return len(df.Gap[df.Gap > 19*60])

In [24]:
total_delay = []
number_bunch = []
number_gap = []

for row in range(main_df.shape[0]):
    time_start = main_df['time period start'].iloc[row]
    time_end = main_df['time period end'].iloc[row]
    E_or_W = main_df['EB/WB'].iloc[row]
    current_data = cleaned_df[(cleaned_df['scheduled time'] >= time_start)
                            & (cleaned_df['scheduled time'] < time_end)
                            & (cleaned_df['EB/WB'] == E_or_W)]

    number_bunch.append(count_bunch(current_data))

    number_gap.append(count_gap(current_data))

# We will not add all the times the streetcar was ahead of schedule
    total_delay.append(sum(current_data['min delay'].apply(lambda x: max(x,0))))

main_df['bunch'] = number_bunch
main_df['total delay'] = total_delay
main_df['gap'] = number_gap

In [25]:
# Finally, drop the 'Gap' column.
if 'Gap' in main_df.columns:
    main_df.drop('Gap', axis=1, inplace=True)

Since our data is big, we split the dataset into time of day/ weekday vs weekend.

In [26]:
# name of the time period column (it's very long)
TIME = [x for x in summary_df.columns if 'Time' in x][0]
# name of the week/sat/sun column (it's also very long)
WEEK = [x for x in summary_df.columns if 'Week' in x][0]

week_translator = {0: 'weekday', 1: 'saturday', 2: 'sunday'}

df_split = dict()
for time in range(5):
    for week in range(3):
        df_split[time, week] = main_df[(main_df[TIME] == time) & (main_df[WEEK] == week)]

Finally, let's write all the data into their own csv files. The final files are not that large because
the rows are split by date/time period.

In [27]:
for key, df in df_split.items():
    to_csv(df,f'../data/schedule_data/processed_data/by_weekday/schedule_data_{week_translator[key[1]]}_{key[0]}.csv')

# Adding a 'by date' version of the data.
Here we create a separate data file that forms a 'by date' version of the data. Since the service periods are removed, we choose to take an average or take a sum over the day depending on the feature.

In [28]:
# First we make sure that date is a timestamp.
main_df['date'] = main_df['date'].apply(lambda x: pd.Timestamp(x))

In [29]:
daily_df_E = pd.DataFrame()
daily_df_W = pd.DataFrame()
def average_feature(date, EB_or_WB, feature, df = main_df):
    length = len(df[(df['date'] == date)
                    & (df['EB/WB'] == EB_or_WB)][feature])
    if length==0:
        return 0
    else:
        return sum(df[(df['date'] == date) \
                      & (df['EB/WB'] == EB_or_WB)][feature])/length

def count_feature(date, EB_or_WB, feature, df = main_df):
    if len(df[(df['date'] == date) & (df['EB/WB'] == EB_or_WB)][feature].tolist())==0:
        return 0
    else:
        return sum(df[(df['date'] == date)
                      & (df['EB/WB'] == EB_or_WB)][feature])

# Separate function to deal with interruptions
def count_interruption(date, EB_or_WB, df = main_df):
    if len(df[(df['date'] == date)
              & (df['EB/WB'] == EB_or_WB)]['Interruption']) == 0:
        return 0
    else:
        return max(df[(df['date'] == date)
                      & (df['EB/WB'] == EB_or_WB)]['Interruption'])


In [30]:
first_date = main_df['date'].min()
last_date = main_df['date'].max()

daily_df_E['date'] = [first_date+pd.Timedelta(days=k) for k in range(352)]
daily_df_E['EB/WB'] = ['E' for _ in range(352)]
daily_df_W['date'] = [first_date+pd.Timedelta(days=k) for k in range(352)]
daily_df_W['EB/WB'] = ['W' for _ in range(352)]

daily_df = pd.concat([daily_df_E, daily_df_W])
daily_df.sort_values('date', inplace=True)
print(daily_df.head())

        date EB/WB
0 2025-01-01     E
0 2025-01-01     W
1 2025-01-02     W
1 2025-01-02     E
2 2025-01-03     E


In [31]:
# Make a choice of which columns are counted and which are averaged
count_columns = ['bunch', 'gap', 'total delay']
average_columns = ['No. of Veh', 'Run time (min)', 'Term time (min)', 'Avg. spd (km/h)', 'RT dist (km)']

In [32]:
for column in count_columns:
    daily_df[column] = daily_df.apply(lambda x: count_feature(x['date'], x['EB/WB'], column), axis=1)

for column in average_columns:
    main_df[column] = main_df[column].apply(lambda x: float(x))
    daily_df[column] = daily_df.apply(lambda x: average_feature(x['date'], x['EB/WB'], column), axis=1)

daily_df['Interruption'] = daily_df.apply(lambda x: count_interruption(x['date'], x['EB/WB']), axis=1)

daily_df.head()

,date,EB/WB,bunch,gap,total delay,No. of Veh,Run time (min),Term time (min),Avg. spd (km/h),RT dist (km),Interruption
0,2025-01-01,E,387,258,2710.900000,16.4,153.4,10.6,11.92,30.13,0
0,2025-01-01,W,94,63,922.533333,16.4,153.4,10.6,11.92,30.13,0
1,2025-01-02,W,52,107,1019.733333,16.4,153.4,10.6,11.92,30.13,0
1,2025-01-02,E,277,313,2556.550000,16.4,153.4,10.6,11.92,30.13,0
2,2025-01-03,E,243,425,3805.383333,16.4,153.4,10.6,11.92,30.13,0


In [33]:
to_csv(daily_df,f'../data/schedule_data/processed_data/by_day/schedule_data.csv')